<a href="https://colab.research.google.com/github/Joselala123/Portugal_House_DataCleaning_EDA_PredictiveModel/blob/main/Portugal_House_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Real Estate Listings in Portugal - Andrade José <br>**
<h2>Data Cleaning, EDA and Predictive Model</h2>


###Utility Functions

In [93]:
!mkdir -p src

In [94]:
%%writefile src/utils.py

import pandas as pd
from IPython.display import display


def display_base_columns(df):
    """
    Displays only the original dataset columns.
    """
    display(df[['Price', 'District', 'City', 'Town', 'EnergyCertificate', 'Floor',
       'NumberOfBathrooms', 'Parking', 'TotalArea', 'ConstructionYear',
       'EnergyEfficiencyLevel', 'PublishDate', 'Garage', 'Elevator',
       'ElectricCarsCharging', 'TotalRooms', 'NumberOfBedrooms', 'NumberOfWC',
       'ConservationStatus', 'LivingArea', 'LotSize', 'BuiltArea']])


def show_nulls(df):
    """
    Displays the number and percentage of missing values
    for each variable.
    """
    nulls = (
        pd.DataFrame({
            "nMissing": df.isna().sum(),
            "percentage": (df.isna().sum() / len(df)) * 100
        })
        .sort_values("percentage", ascending=False)
    )

    print("Count of null values for each variable and its percentage:")
    print()
    display(nulls)


def create_groups(
    df,
    group_columns,
    group_col,
    count_col,
    start_group_at_one=True
):
    """
    Groups observations using the specified columns.

    Creates a unique group ID and counts how many
    observations belong to each group.
    """

    df = df.copy()

    groups = df.groupby(
        group_columns,
        dropna=False
    )

    group_ids = groups.ngroup()
    group_counts = groups[group_columns[0]].transform("size")

    if start_group_at_one:
        group_ids += 1

    df[group_col] = group_ids
    df[count_col] = group_counts

    return df


def assign_duplicate_confidence(
    df,
    condition,
    confidence_level
):
    """
    Assigns a confidence level to observations that
    satisfy the specified condition.
    """

    df.loc[
        condition,
        "DuplicateConfidence"
    ] = confidence_level

    return df

Overwriting src/utils.py


##Data Cleaning

###Libraries and dataset loading

In [116]:
#Libraries for data analysis and visualization
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [117]:
from src.utils import *

In [118]:
# pandas library config
pd.set_option('display.max_rows', None)
pd.set_option("display.max_columns", None)

In [119]:
# Load of dataset

file_id = "1Zb1wlWyr7DQOftR-yf8UsrsK71Gd6bQv"

url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url, low_memory=False)

print(f"Dataset shape: {df.shape}")

Dataset shape: (179001, 23)


###First look of the dataset

In [120]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179001 entries, 0 to 179000
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Price                  178547 non-null  float64
 1   District               179001 non-null  object 
 2   City                   179001 non-null  object 
 3   Town                   178999 non-null  object 
 4   Type                   178985 non-null  object 
 5   EnergyCertificate      178987 non-null  object 
 6   Floor                  27929 non-null   object 
 7   NumberOfBathrooms      162517 non-null  float64
 8   Parking                178969 non-null  float64
 9   TotalArea              164059 non-null  float64
 10  ConstructionYear       117542 non-null  float64
 11  EnergyEfficiencyLevel  110754 non-null  object 
 12  PublishDate            71260 non-null   object 
 13  Garage                 110754 non-null  object 
 14  Elevator               178969 non-nu

In [121]:
df.head()

,Price,District,City,Town,Type,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
0,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,1st Floor,0.0,1.0,114.0,NaN,NaN,NaN,NaN,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN
1,158000.0,Faro,Portimão,Portimão,Apartment,D,2nd Floor,2.0,1.0,21953.0,1993.0,NaN,NaN,NaN,True,NaN,1.0,NaN,NaN,NaN,42.0,NaN,48.0
2,19000.0,Faro,Lagoa (Algarve),Estômbar e Parchal,Land,NC,NaN,0.0,0.0,3710.0,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,3710.0,NaN,3710.0
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,1st Floor,2.0,1.0,115.0,NaN,NaN,NaN,NaN,True,NaN,2.0,NaN,NaN,NaN,115.0,NaN,NaN
4,2950000.0,Faro,Albufeira,Albufeira e Olhos de Água,Building,NC,NaN,0.0,0.0,265.0,1998.0,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,406.0,NaN,465.0


### 1. Basic normalization and house selection

In [122]:
# Normalize the column names
df.columns = df.columns.str.strip()

# Normalize string variables
column_list=["Type", "District", "Town", "City", "EnergyEfficiencyLevel", "EnergyCertificate"]

for col in column_list:
  df[col] = df[col].str.title()

# Assign the correct data type to each variable

text_cols = ["District", "City", "Town", "Type", "EnergyCertificate", "Floor","EnergyEfficiencyLevel", "EnergyEfficiencyLevel","PublishDate","ConservationStatus"]

for col in text_cols:
    df[col] = (df[col].astype("string").str.strip())

int_cols= ["NumberOfBathrooms","ConstructionYear","NumberOfBedrooms","NumberOfWC", "Parking"]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

bool_cols=["Garage", "Elevator", "ElectricCarsCharging"]

for col in bool_cols:
  df[col] = df[col].astype("boolean")

df["PublishDate"] = pd.to_datetime(
    df["PublishDate"],
    format="mixed",
    errors="coerce"
)

# Filter only houses
df = df[df["Type"].eq("House")].copy()

# Drop Type column
df.drop(columns=["Type"], inplace=True)

# register the original position of each observation
df["original_row_id"] = df.index

columns = ["original_row_id"] + [col for col in df.columns if col != "original_row_id"]

df = df[columns].copy()
original_columns=df.columns.drop("original_row_id")

#Add an identifier to each observation
df.insert(len(df.columns), "ID", range(1, len(df) + 1))
df.set_index('ID', inplace=True)

print(f"Number of rows after filter: {len(df)}")

Number of rows after filter: 48744


###2. Missing value inspection

In [123]:
show_nulls(df[original_columns])

Count of null values for each variable and its percentage:



,nMissing,percentage
Floor,47677,97.811013
ConservationStatus,39233,80.487855
PublishDate,29164,59.830954
NumberOfWC,26527,54.421057
LotSize,25801,52.931643
BuiltArea,24464,50.188741
NumberOfBedrooms,18720,38.404727
ElectricCarsCharging,18717,38.398572
Garage,18717,38.398572
EnergyEfficiencyLevel,18717,38.398572


In [124]:
#Variables with a large number of missing values are removed.
#df.drop(columns=["Floor","ConservationStatus", "BuiltArea"], inplace=True)

#### 2.1 Missing-value pattern
A missing value pattern was identified, 18.717 observations have missing values in the same variables, including Garage, ElectricCarsCharging, NumberOfBedrooms, NumberOfWC, LotSize, and PublishDate. Since the dataset was built through web scraping:
*   One possible explanation is that **different scraping sources provide different house characteristics.**


In [125]:
# Variables that belong to the pattern
pattern_columns = [
    "EnergyEfficiencyLevel",
    "PublishDate",
    "Garage",
    "ElectricCarsCharging",
    "NumberOfBedrooms",
    "NumberOfWC",
    "LotSize"
]

In [126]:
df[df[pattern_columns].isna().all(axis=1)].head()

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
1,10,235000.0,Faro,Faro,Faro (Sé E São Pedro),Nc,<NA>,2,0,63.0,<NA>,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,63.0,NaN,NaN
2,26,135000.0,Faro,Vila Real De Santo António,Vila Nova De Cacela,Nc,<NA>,1,1,199.0,1951,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,35.0,NaN,43.0
3,29,199000.0,Faro,Portimão,Portimão,C,<NA>,2,0,82.0,2023,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,82.0,NaN,NaN
4,32,175000.0,Faro,Faro,Faro (Sé E São Pedro),Nc,<NA>,2,0,78.0,1950,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,78.0,NaN,NaN
5,33,1485000.0,Faro,Loulé,Quarteira,D,Ground Floor,5,2,472.0,2004,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,414.0,NaN,NaN


In [127]:
df[df[pattern_columns].isna().all(axis=1)].tail()

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
18717,68239,295000.0,Setúbal,Montijo,Pegões,E,<NA>,1,3,86.0,1982,<NA>,NaT,<NA>,False,<NA>,3.0,<NA>,<NA>,<NA>,106.0,NaN,NaN
18718,68242,645000.0,Setúbal,Palmela,Quinta Do Anjo,A,<NA>,3,1,223.0,2002,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,169.0,NaN,NaN
18719,68245,55000.0,Viana Do Castelo,Arcos De Valdevez,Sabadim,F,<NA>,0,0,1400.0,1937,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,40.0,NaN,80.0
18720,68249,62000.0,Vila Real,Santa Marta De Penaguião,Lobrigos (São Miguel E São João Baptista) E Sa...,Nc,<NA>,1,0,30.0,1997,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,30.0,NaN,60.0
18721,68251,75000.0,Vila Real,Vila Real,Vila Real,Nc,<NA>,2,0,56.0,1987,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,46.0,NaN,96.0


By observing the first and last 5 observations, it is possible that the missing value pattern appears sequentially across all that chunk of data.

In [128]:
indexes = df.index[df[pattern_columns].isna().all(axis=1)].to_series()

# Identify consecutive blocks
blocks = (
    indexes
    .groupby(indexes.diff().ne(1).cumsum())
    .agg(["min", "max", "count"])
)

display(blocks)

,min,max,count
ID,,,
1,1,18699,18699
2,18702,18713,12
3,18716,18721,6


Across most of the interval: [1, 18721], a consistent missing value pattern is observed with a few exceptions. This continuity suggests that the observations in this interval could come from the same scraping process, where the source did not contain the variables that appear as missing.

**Note:** the discussed chunk of data is most likely one source from the scraping process, therefore from now on it will be treated as a real source even though it cant be 100% proved.

In [129]:
#Values that do not follow the pattern
pd.concat([df.iloc[18699:18701], df.iloc[18713:18715]], axis=0)

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
18700,68195,62000.0,Aveiro,Ovar,Válega,F,<NA>,1,1,336.0,1950,F,2024-10-21 15:56:26.597,False,False,False,NaN,2,<NA>,<NA>,300.0,NaN,336.0
18701,68197,220000.0,Aveiro,Oliveira Do Bairro,Palhaça,F,<NA>,1,0,154.0,1993,F,2024-10-18 14:27:49.537,True,False,False,NaN,4,1,<NA>,606.0,5982.0,NaN
18714,68231,37000.0,Aveiro,Ílhavo,São Salvador,Nc,<NA>,2,0,140.0,1950,Nc,2024-10-18 16:15:40.007,False,False,False,NaN,3,2,<NA>,90.0,315.0,140.0
18715,68233,250000.0,Aveiro,Murtosa,Monte,C,<NA>,4,1,68.0,1951,C,2024-10-11 15:39:50.397,False,False,False,NaN,3,1,Like new,55.0,NaN,93.0


A new column is created to identify this new sub block.

In [130]:
# It is enough for data to be present in any of these fields
df["source"] = df[pattern_columns].notna().any(axis=1).map({
    False: "source_A",
    True: "unknown"
})

# Check the classification
summary = (
    df["source"]
    .value_counts()
    .to_frame("records")
    .sort_index()
)

summary["percentage"] = (
    summary["records"] / len(df) * 100
).round(2)

print(summary)

          records  percentage
source                       
source_A    18717        38.4
unknown     30027        61.6


Almost 40% of the filtered dataset belongs to `source_A`.

In [131]:
total_district_counts = df["District"].value_counts()

dfp=pd.concat(
    [
        (df.loc[df["source"]=="source_A", "District"].value_counts()/total_district_counts).rename("source_A_proportion"),
        (df.loc[df["source"]=="unknown", "District"].value_counts()/total_district_counts).rename("unknown_proportion")
    ],
    axis=1
    )

dfp["DistrictTotal"]=total_district_counts
display(dfp.sort_values(by= "DistrictTotal", ascending=False))

,source_A_proportion,unknown_proportion,DistrictTotal
District,,,
Lisboa,0.382857,0.617143,8050
Porto,0.386075,0.613925,6061
Braga,0.391931,0.608069,5478
Setúbal,0.364777,0.635223,3827
Leiria,0.38412,0.61588,3413
Coimbra,0.381039,0.618961,3291
Santarém,0.373872,0.626128,3215
Aveiro,0.394435,0.605565,2875
Faro,0.353663,0.646337,2607


The upper table shows that source: `source_A`, contains a big part of the information about each District.


Even though `source_A` has empty columns we shoudn't just discard it because of the missing columns.
<br></br>
**Note:** linear models usually can't work with missing values, hence if the source is excluded, the model could potentially lose representation from one of the scraping sources, biasing the model.

In [132]:
show_nulls(df[df["source"]!="source_A"])

Count of null values for each variable and its percentage:



,nMissing,percentage
Floor,30027,100.000000
ConservationStatus,20516,68.325174
TotalRooms,16728,55.709861
BuiltArea,15170,50.521198
PublishDate,10447,34.792021
NumberOfWC,7810,26.009924
LotSize,7084,23.592100
ConstructionYear,1123,3.739967
Price,89,0.296400
NumberOfBathrooms,86,0.286409


#### 2.2 Canonical Variables
Further exploration with `source_A` identified revealed that some fields appear to represent different concepts across different sources.

For example: `TotalRooms` in `source_A` frequently matches `NumberOfBedrooms` in the `"unknown"` sources.
<br></br>
These patterns support creating two canonical variables: `bedrooms_Canonical` and `has_parking_Canonical`.

**Note:** Canonical variables are variables that represent the same concept across different data structures, in our case `source_A` and `unknown`.


In [139]:
# Select strong match columns
match_columns = [
    "Price",
    "District",
    "City",
    "Town",
    "LivingArea",
    "ConstructionYear"
]

comparison_df = (
    df.drop_duplicates(subset=original_columns)
    .dropna(subset=match_columns)
    .copy()
)

# Require positive prices and living areas
comparison_df = comparison_df[
    comparison_df[["Price", "LivingArea"]].gt(0).all(axis=1)
]

# Separate the source from the rest of the df.
a = comparison_df[comparison_df["source"].eq("source_A")]
b = comparison_df[comparison_df["source"].eq("unknown")]

# Exclude ambiguous combinations
a = a.drop_duplicates(subset=match_columns, keep=False)
b = b.drop_duplicates(subset=match_columns, keep=False)

# Match records
pairs = a.merge(b, on=match_columns, suffixes=("_A", "_unknown"), validate="one_to_one")

print(f"Cross-schema comparison pairs: {len(pairs):,}")

Cross-schema comparison pairs: 7,914


In [140]:
# Compare parking availability across schemas
pairs["ParkingAvailable_A"] = (pairs["Parking_A"].gt(0))

pairs["ParkingOnly_unknown"] = (pairs["Parking_unknown"].gt(0))

pairs["ParkingOrGarage_unknown"] = (pairs["ParkingOnly_unknown"] | pairs["Garage_unknown"])

# Compare the proposed mappings with alternative interpretations
comparisons = [
    ("TotalRooms_A", "NumberOfBedrooms_unknown"),
    ("TotalRooms_A", "TotalRooms_unknown"),
    ("ParkingAvailable_A", "ParkingOnly_unknown"),
    ("ParkingAvailable_A", "ParkingOrGarage_unknown")
]

results = []

for left, right in comparisons:
    # Missing values do not count as matching evidence
    valid = pairs[left].notna() & pairs[right].notna()
    matches = pairs.loc[valid, left].eq(pairs.loc[valid, right])

    results.append({
        "Comparison": f"{left} = {right}",
        "Comparable pairs": int(valid.sum()),
        "Matching pairs": int(matches.sum()),
        "Match rate (%)": matches.mean() * 100
    })

display(pd.DataFrame(results).round(2))

,Comparison,Comparable pairs,Matching pairs,Match rate (%)
0,TotalRooms_A = NumberOfBedrooms_unknown,7912,7844,99.14
1,TotalRooms_A = TotalRooms_unknown,3818,332,8.70
2,ParkingAvailable_A = ParkingOnly_unknown,7914,5427,68.57
3,ParkingAvailable_A = ParkingOrGarage_unknown,7914,7903,99.86


The table provides strong evidence for creating canonical variables:

* `TotalRooms_A` matches `TotalRooms_unknown` in only approximately **8%** of comparable pairs. Meanwhile, it matches `NumberOfBedrooms_unknown` in approximately **99%** of comparable pairs.

* `ParkingAvailable_A` matches `ParkingOnly_unknown` in approximately **68%** of comparable pairs, but the percentage increases to approximately **99%** when compared with `ParkingOrGarage_unknown`.

These patterns suggest that some fields have different meanings across the two data structures.

`TotalRooms_A` appears to represent the **number of bedrooms**, while the expanded structure (`unknown`) records bedrooms and total rooms separately.

Similarly, `ParkingAvailable_A` appears to represent the availability of **parking or garage**, while the unknown structure records these features separately.

These findings support creating canonical variables with consistent meanings across both structures.

In [141]:
condition = df["source"].eq("source_A")

# Canonical bedrooms
df["bedrooms_Canonical"] = df["NumberOfBedrooms"]

df.loc[condition, "bedrooms_Canonical"] = df.loc[condition, "TotalRooms"]

# Canonical parking or garage availability
parking = df["Parking"].gt(0)
garage = df["Garage"]

df["has_parking_Canonical"] = parking.where(
    condition,
    parking | garage
)

In [142]:
# Compare missing values before and after harmonization
missing_summary = pd.DataFrame({
    "Original field": ["NumberOfBedrooms", "Parking"],
    "Canonical field": ["bedrooms_Canonical", "has_parking_Canonical"],
    "Missing before": [
        df["NumberOfBedrooms"].isna().sum(),
        df["Parking"].isna().sum()
    ],
    "Missing after": [
        df["bedrooms_Canonical"].isna().sum(),
        df["has_parking_Canonical"].isna().sum()
    ]
})

missing_summary["Reduction in missing values"] = (
    missing_summary["Missing before"] - missing_summary["Missing after"]
)

display(missing_summary)

,Original field,Canonical field,Missing before,Missing after,Reduction in missing values
0,NumberOfBedrooms,bedrooms_Canonical,18720,11,18709
1,Parking,has_parking_Canonical,6,6,0


The canonical variables are important throughout whole the project. Both `bedrooms_Canonical` and `has_parking_Canonical` have solid meanings across the two data structures, reducing the risk that differences in how the original fields were recorded could biase the analysis.


### 3. Duplicates

####3.1 Exact duplicates

In [ ]:
dfDuplicates=df.copy()

In [ ]:
print(f"There are {dfDuplicates[original_columns].duplicated().sum()} duplicates in the dataframe.")

There are 2568 duplicates in the dataframe.


In [ ]:
dfDuplicates = create_groups(
    df=dfDuplicates,
    group_columns=original_columns.to_list(),
    group_col="DuplicateGroup",
    count_col="DuplicateCount",
    start_group_at_one=True
)

In [ ]:
dfDuplicates["DuplicateCount"].value_counts().sort_index(ascending=True)

,count
DuplicateCount,
1,44206
2,3412
3,513
4,176
5,60
6,48
7,42
8,32
9,27


The table above shows that some records have more than one associated duplicate, there is even a group of 29 records that are identical in the variables analyzed.<br></br>
**Possible explanations:**
* Some records have the same house design and were published by the same person, organization, among others. This could occur in residential developments.
* Repeatedly capturing a listing that remains published.
* The same property appearing in different scraping sources.
* Concatenating extractions without controlling for records that had already been incorporated.

In [ ]:
print("Top 3 most repeated observations:")
print("\n")
# Get one representative row for each DuplicateGroup, then sort by DuplicateCount
# in descending order and select the top 3.
display_base_columns(dfDuplicates.groupby("DuplicateGroup").first().sort_values(by="DuplicateCount", ascending=False).head(3))

Top 3 most repeated observations:




,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
DuplicateGroup,,,,,,,,,,,,,,,,,,,,,,
32781,455000.0,Braga,Braga,Palmeira,B-,<NA>,4,1,420.0,<NA>,<NA>,NaT,<NA>,False,<NA>,3.0,<NA>,<NA>,<NA>,415.0,NaN,NaN
35288,545000.0,Aveiro,Aveiro,Esgueira,A+,<NA>,3,3,322.0,2023,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,202.0,NaN,265.0
25144,300000.0,Setúbal,Palmela,Palmela,A+,<NA>,2,0,153.0,2023,A+,NaT,True,False,False,4.0,3,1,New,112.0,174.0,153.0


The main limitation here is that there is not enough contextual information to  distinguish different properties from true duplicate records. Further on, I will evaluate model performance both with and without potential duplicates, ensuring that no data leakage occurs.

#### Idea 1


In [ ]:
strong_vars = [
    "Price",
    "Elevator",
    "District",
    "City",
    "Town",
    "LivingArea",
    "ConstructionYear",
    "NumberOfBathrooms"
]

In [ ]:
# Keep rows where all strong variables are available
mask = dfDuplicates[strong_vars].notna().all(axis=1) & (dfDuplicates["DuplicateCount"] == 1)

near_duplicates = dfDuplicates[mask].copy()

# Group observations that match exactly on all strong variables
near_duplicates["NearDupGroup"] = (
    near_duplicates
    .groupby(strong_vars, dropna=False)
    .ngroup()
)

# Keep only groups containing at least 2 observations
group_size = (
    near_duplicates
    .groupby("NearDupGroup")["NearDupGroup"]
    .transform("size")
)

near_duplicates = near_duplicates[
    group_size > 1
].copy()

print("Candidate groups:", near_duplicates["NearDupGroup"].nunique())
print("Candidate observations:", len(near_duplicates))

Candidate groups: 7645
Candidate observations: 15785


In [ ]:
conflict_vars = [
    "PublishDate",
    "TotalArea",
    "BuiltArea",
    "LotSize",
    "NumberOfBedrooms",
    "TotalRooms",
    "NumberOfWC",
    "Parking",
    "Garage",
    "ElectricCarsCharging",
    "EnergyCertificate",
    "ConservationStatus"
]

In [ ]:
conflict_columns = []

for col in conflict_vars:

    conflict_col = f"{col}Conflict"

    # Count distinct NON-NULL values inside each group
    n_unique = (
        near_duplicates
        .groupby("NearDupGroup")[col]
        .transform(lambda x: x.nunique(dropna=True))
    )

    # Conflict exists when there is more than one observed value
    near_duplicates[conflict_col] = n_unique > 1

    conflict_columns.append(conflict_col)

In [ ]:
near_duplicates["ConflictCount"] = (
    near_duplicates[conflict_columns]
    .sum(axis=1)
)

In [ ]:
conflict_columns = [
    "PublishDateConflict",
    "TotalAreaConflict",
    "BuiltAreaConflict",
    "LotSizeConflict"
]

In [ ]:
near_duplicates["ConflictCount"] = (
    near_duplicates[conflict_columns]
    .sum(axis=1)
)

near_duplicates["ConflictCount"].value_counts()

,count
ConflictCount,
1,10345
0,3491
2,1842
3,93
4,14


In [ ]:
conflicts_by_variable = (
    near_duplicates
    .drop_duplicates("NearDupGroup")[conflict_columns]
    .sum()
    .sort_values(ascending=False)
)

display(conflicts_by_variable)

,0
TotalAreaConflict,5410
BuiltAreaConflict,876
PublishDateConflict,401
LotSizeConflict,129


In [ ]:
near_duplicates[near_duplicates["PublishDateConflict"]==True].head()

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source,DuplicateGroup,DuplicateCount,NearDupGroup,PublishDateConflict,TotalAreaConflict,BuiltAreaConflict,LotSizeConflict,NumberOfBedroomsConflict,TotalRoomsConflict,NumberOfWCConflict,ParkingConflict,GarageConflict,ElectricCarsChargingConflict,EnergyCertificateConflict,ConservationStatusConflict,ConflictCount
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
297,1554,895000.0,Faro,Albufeira,Albufeira E Olhos De Água,C,<NA>,5,3,670.0,2023,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,210.0,NaN,NaN,source_A,41324,1,30402,True,True,False,True,False,True,False,True,False,False,False,False,3
652,2738,895000.0,Faro,Albufeira,Albufeira E Olhos De Água,C,<NA>,5,3,499.0,2023,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,210.0,NaN,NaN,source_A,41323,1,30402,True,True,False,True,False,True,False,True,False,False,False,False,3
692,2897,895000.0,Faro,Albufeira,Albufeira E Olhos De Água,C,<NA>,5,3,453.0,2023,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,210.0,NaN,NaN,source_A,41322,1,30402,True,True,False,True,False,True,False,True,False,False,False,False,3
1185,4880,695000.0,Faro,Vila Do Bispo,Sagres,C,<NA>,3,0,234.0,2008,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,176.0,NaN,NaN,source_A,38619,1,28520,True,True,False,False,False,False,False,False,False,False,False,False,2
1753,6849,159900.0,Lisboa,Azambuja,Azambuja,E,<NA>,1,0,113.0,1937,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,73.0,NaN,82.0,source_A,14982,1,11033,True,True,False,False,False,True,False,False,False,False,False,False,2


####3.2 Near-duplicates
While I was exploring the dataset, I noticed that **many records had the same information but with minor differences**. For example, some records were identical except for the `publishDate`.

As I mentioned before, it is difficult to identify true duplicates .
Therefore, the best strategy for now is to create a new column that indicates how likely each record is to be a duplicate.  



In [ ]:
dfDuplicates["DuplicateConfidence"] = "NotFlagged"

dfDuplicates = assign_duplicate_confidence(
    dfDuplicates,
    ((dfDuplicates["DuplicateCount"] > 1)),
    "Identical"
)

Observations with more than one associated duplicate are flagged as `Identical`.

Now lets analyze values which only difference is `PublishDate`.

In [ ]:
dfDuplicates = create_groups(
    df=dfDuplicates,
    group_columns=original_columns.drop("PublishDate").to_list(),
    group_col="DateGroup",
    count_col="DateGroupCount",
    start_group_at_one=True
)

In [ ]:
date_duplicate_candidates = dfDuplicates[
    (dfDuplicates["DuplicateCount"] == 1) &
    (dfDuplicates["DateGroupCount"] > 1)
].sort_values(
    ["DateGroup", "PublishDate"]
)

display_base_columns(date_duplicate_candidates.head())

,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,
22240,10000.0,Coimbra,Oliveira Do Hospital,Seixo Da Beira,Nc,<NA>,1,0,44.0,2007,Nc,2024-06-11 08:37:29.240,False,False,False,NaN,2,1,<NA>,44.0,72.0,72.0
39338,10000.0,Coimbra,Oliveira Do Hospital,Seixo Da Beira,Nc,<NA>,1,0,44.0,2007,Nc,NaT,False,False,False,NaN,2,1,<NA>,44.0,72.0,72.0
35144,12000.0,Viana Do Castelo,Ponte Da Barca,Lindoso,Nc,<NA>,0,0,45.0,1937,Nc,2023-05-08 17:14:42.000,False,False,False,NaN,1,0,Used,45.0,45.0,90.0
30725,12000.0,Viana Do Castelo,Ponte Da Barca,Lindoso,Nc,<NA>,0,0,45.0,1937,Nc,NaT,False,False,False,NaN,1,0,Used,45.0,45.0,90.0
38541,20000.0,Castelo Branco,Castelo Branco,São Vicente Da Beira,Nc,<NA>,1,0,42.0,1937,Nc,2026-05-26 09:50:08.457,False,False,False,NaN,2,<NA>,Needs renovation,42.0,NaN,NaN


In [ ]:
date_group_summary = (
    date_duplicate_candidates
    .groupby("DateGroup")
    .agg(
        records=("DateGroup", "size"),
        valid_dates=("PublishDate", "count"),
        unique_dates=("PublishDate", "nunique"),
        first_date=("PublishDate", "min"),
        last_date=("PublishDate", "max"),
        Date_diff=("PublishDate", lambda x: x.max() - x.min())
    )
)
date_group_summary.sort_values("records", ascending=False)

,records,valid_dates,unique_dates,first_date,last_date,Date_diff
DateGroup,,,,,,
35036,31,30,30,2024-05-03 10:30:56.123,2024-05-03 11:16:29.797,0 days 00:45:33.674000
4052,21,21,21,2024-12-13 16:10:44.137,2024-12-13 16:44:59.953,0 days 00:34:15.816000
36888,14,14,14,2025-11-20 17:03:56.820,2025-11-20 17:26:38.210,0 days 00:22:41.390000
45205,11,10,10,2024-11-15 17:40:25.640,2024-11-15 17:41:09.143,0 days 00:00:43.503000
26056,11,11,11,2025-01-16 18:02:54.063,2025-01-16 18:19:56.420,0 days 00:17:02.357000
36695,10,10,10,2024-04-30 18:29:14.980,2024-05-03 11:33:34.057,2 days 17:04:19.077000
18676,9,9,9,2026-03-26 09:50:07.543,2026-08-22 08:28:54.470,148 days 22:38:46.927000
34185,9,9,9,2024-08-23 15:17:37.923,2024-08-23 16:31:03.167,0 days 01:13:25.244000
26948,8,7,7,2025-01-27 12:50:58.880,2025-01-27 12:52:45.013,0 days 00:01:46.133000


In [ ]:
# Groups where distance can actually be evaluated
comparable_groups = date_group_summary[
    date_group_summary["valid_dates"] >= 2
]

print("Total groups:", len(date_group_summary))

print(
    "Groups with at least 2 valid dates:",
    len(comparable_groups)
)

print(
    "Groups with date difference < 48h:",
    (comparable_groups["Date_diff"] < pd.Timedelta(hours=48)).sum()
)

print(
    "Proportion:",
    (comparable_groups["Date_diff"] < pd.Timedelta(hours=48)).mean()
)

Total groups: 535
Groups with at least 2 valid dates: 367
Groups with date difference < 24h: 214
Proportion: 0.5831062670299727


After restricting the analysis to groups with at least two valid dates, we observe a clear trend, when comparing the first and last `PublishDate`, we found that nearly 60% of the near-duplicates have a time span less than 48 hours between their earliest and latest records. This temporal concentration provides additional evidence that many of these groups may represent repeated captures or updates of the same listing.

### 4. Validation of impossible values

In [ ]:
#Simple inspection of minimum and maximum values
df.describe().loc[['min', 'max']]

,original_row_id,Price,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,PublishDate,TotalRooms,NumberOfBedrooms,NumberOfWC,LivingArea,LotSize,BuiltArea
min,10.0,1.000000e+00,-1.0,0.0,-270.0,1900.0,2018-05-02 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0
max,178995.0,1.380000e+09,3020.0,3.0,5525000.0,2026.0,2026-09-02 20:30:15.870000,2751.0,21.0,3016.0,470300.0,40672000.0,470300.0


In [ ]:
# Price
df = df[(df["Price"] > 0) & (df["Price"] <= 100_000_000)].copy()

# Remove observation with a value of 1.
df.drop(
    index=df[df["Price"] == 1].index,
    inplace=True
)

In [ ]:
# Areas
area_cols = ["LivingArea", "TotalArea", "LotSize"]

for col in area_cols:
    invalid = (df[col].notna() & (df[col] <= 5)) #An area variable with a value of 5 or less is considered impossible

    df.loc[invalid, col] = np.nan

In [ ]:
#Illogical areas (LivingArea > TotalArea)

(((df["LivingArea"] > df["TotalArea"])).sum())

np.int64(6254)

LivingArea and TotalArea do not conceptually represent the same thing; however, there are 4,247 observations that appear this way. <br>
We will visualize how the difference between LivingArea and TotalArea behaves.

In [ ]:
mask = (df["LivingArea"].notna() & df["TotalArea"].notna() & (df["LivingArea"] > df["TotalArea"]))

area_review = df.loc[mask, ["LivingArea", "TotalArea"]].copy()

area_review["difference"] = (area_review["LivingArea"] - area_review["TotalArea"])

area_review["ratio"] = (area_review["LivingArea"] / area_review["TotalArea"])

area_review[["difference", "ratio"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])

,difference,ratio
count,6254.000000,6254.000000
mean,335.284938,3.022522
std,5514.776580,30.186881
min,1.000000,1.000812
10%,15.000000,1.109756
25%,30.000000,1.250000
50%,63.000000,1.571429
75%,117.000000,2.000000
90%,200.000000,2.625000
95%,300.000000,3.186667


There are considerable differences between LivingArea and TotalArea in a relevant proportion of observations. These inconsistencies could be due to differences in how the variables are defined across scraping sources, recording errors, user confusion, among other reasons. <br>
Since it is not possible to determine which of the two variables is correct in each case, the observations will be retained and an indicator variable will be created to identify cases where LivingArea > TotalArea. This variable will allow further analysis and could help model performance.

In [ ]:
mask = (df["TotalArea"].notna() & df["LivingArea"].notna())

df["AreaInconsistency"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

df.loc[mask, "AreaInconsistency"] = (df.loc[mask, "LivingArea"]> df.loc[mask, "TotalArea"]).astype(int)

In [ ]:
df[['TotalArea','LivingArea',"AreaInconsistency"]].head()

,TotalArea,LivingArea,AreaInconsistency
ID,,,
1,63.0,63.0,0
2,199.0,35.0,0
3,82.0,82.0,0
4,78.0,78.0,0
5,472.0,414.0,0


In [ ]:
# Count variables
count_cols = [
    "NumberOfBathrooms",
    "NumberOfBedrooms",
    "NumberOfWC",
    "TotalRooms",
]

for col in count_cols:
    invalid = ((df[col].notna())& (df[col] < 0))

    if invalid.any():
        print(f"{col}: {invalid.sum():,} valores negativos convertidos a NaN.")
        df.loc[invalid, col] = np.nan

#Thresholds above which an observation is considered impossible.
limits = {                   #Interpretation:
    "NumberOfBathrooms": 30,  #Number of bathrooms >30
    "NumberOfWC": 30,
    "NumberOfBedrooms": 50,
    "TotalRooms": 100
}

for col, upper in limits.items():
    invalid = (df[col].notna() & (df[col] > upper))

    print(f"{col} > {upper}: {invalid.sum():,} valores convertidos a NaN.")

    df.loc[invalid, col] = np.nan


NumberOfBathrooms: 1 valores negativos convertidos a NaN.
NumberOfBathrooms > 30: 2 valores convertidos a NaN.
NumberOfWC > 30: 1 valores convertidos a NaN.
NumberOfBedrooms > 50: 0 valores convertidos a NaN.
TotalRooms > 100: 1 valores convertidos a NaN.


### 5. Cleaning summary

In [ ]:
print(f"Casas finales después de limpieza: {len(df)}")
print(f"Variables finales: {df.shape[1]}")

missing = (
    pd.DataFrame({"nMissing": df.isna().sum(), "percentage": (df.isna().sum()/len(df))*100})
    .sort_values("percentage", ascending=False)
)
print("Cantidad de varoles nulos por variable y porcentaje:")
display(missing)

print("\nRango de PublishDate:")
valid_publish_dates = df["PublishDate"].dropna()
if not valid_publish_dates.empty:
    print(valid_publish_dates.min()," - " ,valid_publish_dates.max())
else:
    print("No valid PublishDate entries found.")

Casas finales después de limpieza: 48626
Variables finales: 25
Cantidad de varoles nulos por variable y porcentaje:


,nMissing,percentage
Floor,47560,97.807757
ConservationStatus,39165,80.543331
PublishDate,29090,59.823962
NumberOfWC,26473,54.442068
LotSize,25851,53.162917
BuiltArea,24393,50.164521
NumberOfBedrooms,18692,38.440341
EnergyEfficiencyLevel,18689,38.434171
ElectricCarsCharging,18689,38.434171
Garage,18689,38.434171



Rango de PublishDate:
2018-05-02 00:00:00  -  2026-09-02 20:30:15.870000


In [ ]:
# Save clean dataset

#MODIFICATION: new name to distinguish it from the original project's CSV.
OUTPUT_PATH = "portugal_listings_limpio.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Archivo guardado como: {OUTPUT_PATH}"
)


Archivo guardado como: portugal_listings_limpio.csv
